In [ ]:
import numpy as np
import pandas as pd
import pyarrow
import sqlite3

### data loading

In [ ]:

df_arrests: pd.DataFrame = pd.read_csv('/Users/leahmayotte/Documents/education/umass/capstone/2_data/raw/arrests-latest.csv', low_memory=False)
df_detainers: pd.DataFrame  = pd.read_csv('/Users/leahmayotte/Documents/education/umass/capstone/2_data/raw/detainers-latest.csv', low_memory=False)
df_detention_stays: pd.DataFrame = pd.read_csv('/Users/leahmayotte/Documents/education/umass/capstone/2_data/raw/detention_stays-latest.csv', low_memory=False)

print(f"    - {len(df_arrests)} arrests")
print(f"    - {len(df_detainers)} detainers")
print(f"    - {len(df_detention_stays)} detentions")

### data exploration

In [ ]:
df_arrests.info()

In [ ]:
df_detainers.info()

In [ ]:
df_detention_stays.info()

### null handling

In [ ]:
def calculate_null_percentage():

    arrests_null_percentages: float = (df_arrests.isnull().sum() / len(df_arrests) * 100).round(2)
    print("Arrests Table Null Percentages")
    with pd.option_context('display.max_rows', None):
        print(arrests_null_percentages)
    print()
    print()

    detainers_null_percentages = (df_detainers.isnull().sum() / len(df_detainers) * 100).round(2)
    print("Detainers Table Null Percentages")
    with pd.option_context('display.max_rows', None):
        print(detainers_null_percentages)
    print()
    print()

    detentions_null_percentages = (df_detention_stays.isnull().sum() / len(df_detention_stays) * 100).round(2)
    print("Detention Stays Table Null Percentages")
    with pd.option_context('display.max_rows', None):
        print(detentions_null_percentages)

calculate_null_percentage()

In [ ]:
# find outlier Null values in each table
print('Scanning for Suspicious Null Values...')

# create a list of values that I want to look for
suspicious_null_values: list[str] = ['','-', 'N/A', 'None', 'NULL', 'null', 'none', 'unknown', '--']
# only return column names that are strings
arrests_string_columns: pd.Index = df_arrests.select_dtypes(include='str').columns
detainers_string_columns: pd.Index = df_detainers.select_dtypes(include='str').columns
detentions_string_columns: pd.Index = df_detention_stays.select_dtypes(include='str').columns

# iterate over each column
for column in arrests_string_columns:
    # look for only unique values, dropping NaN values, so only non-null values are inspected
    unique_vals: np.ndarray = df_arrests[column].dropna().unique()
    # as the loop iterates, build a list of values that match the suspicious_null_values defined
    suspicious: list[str] = []
    for value in unique_vals:
        if str(value).strip() in suspicious_null_values:
            suspicious.append(value)
    if suspicious:
        print('Arrests Table: Suspicious Nulls Found')
        print(f'{column}: {suspicious}')

print('Arrests Table: Scan Complete')

for column in detainers_string_columns:
    unique_vals: np.ndarray = df_detainers[column].dropna().unique()
    suspicious: list[str] = []
    for value in unique_vals:
        if str(value).strip() in suspicious_null_values:
             suspicious.append(value)
    if suspicious:
        print('Detainers Table: Suspicious Nulls Found')
        print(f'{column}: {suspicious}')

print('Detainers Table: Scan Complete')

for column in detentions_string_columns:
    unique_vals: np.ndarray = df_detention_stays[column].dropna().unique()
    suspicious: list[str] = []
    for value in unique_vals:
        if str(value).strip() in suspicious_null_values:
            suspicious.append(value)
    if suspicious:
        print('Detention Stays Table: Suspicious Nulls Found')
        print(f'{column}: {suspicious}')

print('Detention Stays Table: Scan Complete')


### removing unnecessary data

In [ ]:
# check for existing columns
print(df_arrests.columns.tolist())
print()
print(df_detainers.columns.tolist())
print()
print(df_detention_stays.columns.tolist())

In [ ]:
# create lists naming the columns to be removed
arrests_cols_to_drop: list[str] = [
    # 100% the same value
    'file_original',
    'sheet_original',
    'final_program_group',
]

detainers_cols_to_drop: list[str] = [
    # 100% same value
    'file_original',
    'sheet_original',
    # 100% null
    'unlawful_attempt_yes_no',
    'unlawful_entry_yes_no',
    'federal_interest_yes_no',
    'visa_yes_no',
    # effectively null
    'illegal_entry_yes_no',
    'immigration_fraud_yes_no',
    'other_removal_reason_yes_no',
    # not relevant to analysis
    'port_of_departure',
    'arrest_time_case_category',
    'arrest_time_current_program',
    'msc_charge_date',
    'msc_conviction_date',
    'order_show_cause_served_yes_no',
    'statements_made_yes_no',
]

detention_stays_cols_to_drop: list[str] = [
    # effectively null
    'religion',
    # redundant or not relevant
    'stay_book_out_date',
    'detention_facility_codes_all',
    'book_in_date_time_first',
]

# drop columns specified above
df_arrests = df_arrests.drop(columns=arrests_cols_to_drop)
df_detainers = df_detainers.drop(columns=detainers_cols_to_drop)
df_detention_stays = df_detention_stays.drop(columns=detention_stays_cols_to_drop)

### duplicate handling

In [ ]:
# utilize the existing duplicate_likely column as a flag for duplicates
df_arrests = df_arrests[df_arrests['duplicate_likely'] == 0]
df_detainers = df_detainers[df_detainers['duplicate_likely'] == 0]

# once filtered, drop the 'duplicate_likely' column as it is no longer necessary
df_arrests = df_arrests.drop(columns=['duplicate_likely'])
df_detainers = df_detainers.drop(columns=['duplicate_likely'])

In [ ]:
# recheck columns for each table
print(df_arrests.columns.tolist())
print()
print(df_detainers.columns.tolist())
print()
print(df_detention_stays.columns.tolist())

### data type correction

In [ ]:
# check column data types
df_arrests.dtypes

In [ ]:
df_detainers.dtypes

In [ ]:
df_detention_stays.dtypes

In [ ]:
# convert to DATETIME

arrests_date_cols: list[str] = [
    'apprehension_date',
    'apprehension_date_time',
    'departed_date',
]

detainers_date_cols: list[str] = [
    'detainer_prepare_date',
    'final_order_date',
    'apprehension_date',
    'entry_date',
    'departed_date',
]

detention_stays_date_cols: list[str] = [
    'stay_book_in_date_time',
    'stay_book_out_date_time',
    'final_order_date',
    'bond_posted_date',
    'book_out_date_time_first',
    'book_in_date_time_longest',
    'book_out_date_time_longest',
    'book_in_date_time_last',
    'book_out_date_time_last',
]

for col in arrests_date_cols:
    df_arrests[col] = pd.to_datetime(df_arrests[col])

for col in detainers_date_cols:
    df_detainers[col] = pd.to_datetime(df_detainers[col])

for col in detention_stays_date_cols:
    df_detention_stays[col] = pd.to_datetime(df_detention_stays[col])


In [ ]:
# convert birth year to integer (preserve nulls)
df_arrests['birth_year'] = df_arrests['birth_year'].astype('Int64')
df_detainers['birth_year'] = df_detainers['birth_year'].astype('Int64')

In [ ]:
# standardize text columns
arrests_title_cols: list[str] = [
    'apprehension_state',
    'apprehension_aor',
    'apprehension_method',
    'apprehension_criminality',
    'final_program',
    'case_status',
    'case_category',
    'departure_country',
    'citizenship_country',
    'gender',
]

detainer_title_cols: list[str] = [
    'facility_state',
    'facility_aor',
    'departure_country',
    'case_status',
    'detainer_prepared_criminality',
    'detention_facility',
    'gender',
    'citizenship_country',
    'birth_country',
    'entry_status',
    'most_serious_conviction_charge',
    'felon',
    'processing_disposition',
    'case_category',
    'final_program',
    'detainer_type',
    'detainer_lift_reason',
]

detention_stays_title_cols: list[str] = [
    'gender',
    'citizenship_country',
    'ethnicity',
    'entry_status',
    'felon',
    'case_status',
    'case_category',
    'book_in_criminality',
    'final_charge',
    'departure_country',
    'final_program',
    'detention_release_reason',
    'stay_release_reason',
    'marital_status',
    'msc_charge',
    'detention_facility_first',
    'detention_facility_longest',
    'detention_facility_last',
]

for col in arrests_title_cols:
    df_arrests[col] = df_arrests[col].str.strip().str.upper()

for col in detainer_title_cols:
    df_detainers[col] = df_detainers[col].str.strip().str.upper()

for col in detention_stays_title_cols:
    df_detention_stays[col] = df_detention_stays[col].str.strip().str.upper()

In [ ]:
# # verify text change
# df_arrests.head()

In [ ]:
# df_detainers.head()

In [ ]:
# df_detention_stays.head()

In [ ]:
# convert appropriate binaries to booleans

arrests_bool_cols: list[str] = [
    'final_order_yes_no'
]

detainers_bool_cols: list[str] = [
    'resume_custody_yes_no',
    'biometric_match_yes_no',
    'prior_felony_yes_no',
    'violent_misdemeanor_yes_no',
    'aggravated_felony_yes_no',
    'deportation_ordered_yes_no',
    'final_order_yes_no',
    'case_final_order_yes_no',
]

detention_stays_bool_cols: list[str] = [
    'final_order_yes_no',
]

# recognize YES/NO as TRUE/FALSE
bool_type: dict[str, bool] = {
    'YES' : True,
    'NO' : False
}

for col in arrests_bool_cols:
    df_arrests[col] = df_arrests[col].map(bool_type).astype('boolean')

for col in detainers_bool_cols:
    df_detainers[col] = df_detainers[col].map(bool_type).astype('boolean')

for col in detention_stays_bool_cols:
    df_detention_stays[col] = df_detention_stays[col].map(bool_type).astype('boolean')

# check that it worked
print(df_arrests['final_order_yes_no'].value_counts(dropna=False))
print(df_arrests['final_order_yes_no'].dtype)


### data creation for analysis

In [ ]:
# simplifying criminality data for analysis
criminality_type: dict[str, str] = {
    '1 CONVICTED CRIMINAL' :    'CONVICTED CRIMINAL',
    '2 PENDING CRIMINAL CHARGES':   'PENDING CHARGES',
    '3 OTHER IMMIGRATION VIOLATOR':    'OTHER IMMIGRATION VIOLATOR'
}

df_arrests['criminality_clean'] = df_arrests['apprehension_criminality'].map(criminality_type)
df_detainers['criminality_clean'] = df_detainers['detainer_prepared_criminality'].map(criminality_type)
df_detention_stays['criminality_clean'] = df_detention_stays['book_in_criminality'].map(criminality_type)

In [ ]:
# calculate detention length in detention_stays

df_detention_stays['detention_length_days'] = (
    df_detention_stays['stay_book_out_date_time'] -
    df_detention_stays['stay_book_in_date_time']
).dt.days

In [ ]:
# create an 'age' column using 'birth_year' conversion

from datetime import datetime

current_year: int = datetime.now().year

df_arrests['age'] = current_year - df_arrests['birth_year']
df_detainers['age'] = current_year - df_detainers['birth_year']
df_detention_stays['age'] = current_year -df_detention_stays['birth_year']

In [ ]:
# extract year and month from critical date columns
df_arrests['apprehension_year'] = df_arrests['apprehension_date'].dt.year
df_arrests['apprehension_month'] = df_arrests['apprehension_date'].dt.month

df_detainers['detainer_year'] = df_detainers['detainer_prepare_date'].dt.year
df_detainers['detainer_month'] = df_detainers['detainer_prepare_date'].dt.month

df_detention_stays['stay_year'] = df_detention_stays['stay_book_in_date_time'].dt.year
df_detention_stays['stay_month'] = df_detention_stays['stay_book_in_date_time'].dt.month

In [ ]:
# convert sentence to days for comparative duration columns

df_detainers['msc_sentence_days_total'] = (
    df_detainers['msc_sentence_days'].fillna(0) +
    (df_detainers['msc_sentence_months'].fillna(0) * 30.44) +
    (df_detainers['msc_sentence_years'].fillna(0) * 365)
).round(0)

# Null out any row where the total calculated to zero
# covers combinations where days = 0, months = null, years = 0, etc.
zero_total_mask = (df_detainers['msc_sentence_days_total'] == 0)

df_detainers.loc[zero_total_mask, 'msc_sentence_days_total'] = np.nan

print(f'Rows nulled out:    {zero_total_mask.sum():,}')
print(f"Remaining nulls:   {df_detainers['msc_sentence_days_total'].isna().sum():,}")

remaining_zeros = (df_detainers['msc_sentence_days_total'] == 0).sum()
print(f'Remaining zeros:    {remaining_zeros:,}')

# check the distribution of the new 'msc_sentence_days_total' column
print(df_detainers['msc_sentence_days_total'].describe())


### data export

In [ ]:
# export CSV
cleaned_data_path: str = '/Users/leahmayotte/Documents/education/umass/capstone/2_data/cleaned/'

df_arrests.to_csv(f'{cleaned_data_path}arrests_cleaned.csv', index=False)
df_detainers.to_csv(f'{cleaned_data_path}detainers_cleaned.csv', index=False)
df_detention_stays.to_csv(f'{cleaned_data_path}detention_stays_cleaned.csv', index=False)

In [ ]:
# export PARQUET
cleaned_data_path: str = '/Users/leahmayotte/Documents/education/umass/capstone/2_data/cleaned/'

df_arrests.to_parquet(f'{cleaned_data_path}arrests_cleaned.parquet', index=False)
df_detainers.to_parquet(f'{cleaned_data_path}detainers_cleaned.parquet', index=False)
df_detention_stays.to_parquet(f'{cleaned_data_path}detention_stays_cleaned.parquet', index=False)

In [ ]:
# export to SQL DB
conn = sqlite3.connect('/Users/leahmayotte/Documents/education/umass/capstone/5_sql/capstone_database_sql.db')

df_arrests.to_sql('arrests', conn, if_exists='replace', index=False)
df_detainers.to_sql('detainers', conn, if_exists='replace', index=False)
df_detention_stays.to_sql('detentions', conn, if_exists='replace', index=False)

cursor = conn.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
print(cursor.fetchall())

conn.close()
